In [1]:
import pandas as pd
import numbers as np
import category_encoders as ce
from sklearn.preprocessing import StandardScaler, LabelEncoder


df = pd.read_csv('../../dataset/car_prices.csv', on_bad_lines='skip')

In [2]:
import math
import os
from decimal import Decimal, ROUND_HALF_DOWN
brand_mapping = {
            'landrover': 'land rover', 'land rover': 'land rover',
            'mercedes': 'mercedes-benz', 'mercedes-b': 'mercedes-benz',
            'vw': 'volkswagen', 'volkswagen': 'volkswagen',
            'gmc truck': 'gmc', 'gmc': 'gmc',
            'ford truck': 'ford', 'ford tk': 'ford', 'ford': 'ford',
            'dodge tk': 'dodge', 'dodge': 'dodge',
            'chev truck': 'chevrolet', 'chevrolet': 'chevrolet',
            'hyundai tk': 'hyundai', 'hyundai': 'hyundai',
            'mazda tk': 'mazda', 'mazda': 'mazda'
        }
output_dir = '../../dataset'
os.makedirs(output_dir, exist_ok=True)

In [3]:
def categorize_trim(trim):
    if trim is None or (isinstance(trim, float) and math.isnan(trim)):
        return 'other'

    t = str(trim).lower().strip()

    categories = {
        'special edition': [
            'se', 'limited', 'lt', 'le', 'sel', 'ltz', '1lt', 'lt fleet', 'sle', '2lt', 'es', 'lt1',
            '2.5i premium pzev', 'unlimited sahara', '3,5 se', 'sle-1', 'king ranch', 'wolfsburg edition pzev',
            'se v6', 'special edition', 'zx4 se', 'sle-2', 'sle1', '2ss', 'ultimate', 'dx', 'el limited',
            'le v6', 'signature', 'se fleet', 'tdi', 'heat', 'lt3', '2.5i limited', 'sle 1500', '1500 sle',
            'wolfsburg edition', 'passion coupe', 'se1', 'sel plus'
        ],
        'touring': ['touring', 'slt', 'i touring', 'slt-1', 'slt-2', 'touring-l', 'grand touring', 'custom',
                    'lt 1500', 'slt 1500', 'i grand touring', 's grand touring', 'gtp'],
        'luxury': ['lx', 'xlt', 'gls', 'ex', 'ex-l', 'lariat', 'titanium', 'luxury', 'premium', 'xle', '+',
                   'denali', 'cxl', 'xe', '2.0 t premium quattro', 'laramie', 'platinum', 'technology package',
                   'premier', 'deluxe', 'cargo vn xlt', 'lx-p', 'gxe', 'gls 1.8t', 'glk350', 'gl450', 'gle',
                   'lx-s'],
        'sport': ['ls', 's', '2,5 s', 'sport', 'sv', '3,5 sv', 'sl', 'ls fleet', 'gt', '3,5 s', 'i sport', 'xls',
                  'sr5', '1,8 s', 'st', '3.2', '1500 ls', 'gs', '2,0 sr', '3,5 sl', '2.0t', 'sx',
                  'c300 sport 4matic', '2,0 s', 'c300 sport', 'stx', '1,6 s plus', '3.0si', 'sr', 'gt premium',
                  's pzev', 'v8', '1,8 sl', 'performance', 'supercharged', 's sport', 'turbo', 'prerunner v6',
                  '1,6 s', 'quattro', 'ls 3500', '3.5 sr', '2,0 sl', '3.2 quattro', 'sl500', 'gts', 'c350 sport',
                  'xle v6', 'sl 550', 'ex v6', 'sport pzev', 'r350', 'sl2', 'sle v6'],
        'base': ['base', 'sxt', 'xl', 'laredo', 'se pzev', 'r/t', 'hybrid', 'ce', 'american value', 'sxt fleet',
                 'cx', 'ltz fleet', 'tdi', 'crew', 'ltz 1500', 'rt', 'express', 'mainstreet', 'overland', 'eco',
                 'standard', 'fx35', 'fx4', 'fx2', 'comfort', 'value leader', 'convenience', 'easy',
                 'value package', 'journey']
    }

    for category, keywords in categories.items():
        if any(kw in t for kw in keywords):
            return category

    return 'other'

In [4]:
def categorize_body(carrozzeria):
    categories = {
        "sedan": ["sedan", "g sedan"],
        "suv": ["suv"],
        "hatchback": ["hatchback"],
        "van": ["minivan", "van", "e-series van", "transit van", "promaster cargo van", "ram van"],
        "coupé": ["coupe", "g coupe", "genesis coupe", "koup", "cts coupe", "elantra coupe", "q60 coupe",
                  "g37 coupe", "cts-v coupe"],
        "cabriolet": ["convertible", "g convertible", "beetle convertible", "q60 convertible",
                         "g37 convertible"],
        "station wagon": ["wagon", "tsx sport wagon", "cts wagon", "cts-v wagon"],
        "pickup": ["regular cab", "regular-cab", "extended cab", "king cab", "access cab", "xtracab", "cab plus",
                   "cab plus 4", "crew cab", "supercab", "quad cab", "double cab", "crewmax cab", "mega cab", "supercrew"],
    }

    for category, values in categories.items():
        if carrozzeria in values:
            return category

    return "other"

In [5]:
def preprocess(data, output_filename="processed_data.csv"):
    data = data.drop(columns=['vin', 'seller', 'saledate', 'state', 'mmr'], errors='ignore')
    data = data.rename(columns={
        "year": "anno produzione",
        "make": "marca",
        "model": "modello",
        "trim": "allestimento",
        "body": "carrozzeria",
        "transmission": "trasmissione",
        "condition": "condizione",
        "odometer": "chilometraggio",
        "color": "colorazione",
        "interior": "colore interni",
        "sellingprice": "prezzo"
    })

    soglia = 500
    counts = data['marca'].value_counts()
    valori_validi = counts[counts >= soglia].index
    data = data[data['marca'].isin(valori_validi)]

    data = data[data['prezzo'] > 500]

    str_cols = data.select_dtypes(include=['object', 'string']).columns
    data[str_cols] = data[str_cols].apply(lambda x: x.str.lower().str.strip())

    data['marca'] = data['marca'].replace(brand_mapping)

    data['condizione'] = data['condizione'].apply(
        lambda x: int(Decimal(x).to_integral_value(rounding=ROUND_HALF_DOWN)) if pd.notnull(x) else pd.NA
    )

    data['chilometraggio'] = data['chilometraggio'].apply(
        lambda x: int(Decimal(x).to_integral_value(rounding=ROUND_HALF_DOWN)) if pd.notnull(x) else pd.NA
    )

    data.loc[data['allestimento'] == data['modello'], 'allestimento'] = 'base'

    data.loc[data['colorazione'] == '—', 'colorazione'] = None
    data.loc[data['colore interni'] == '—', 'colore interni'] = None

    data['allestimento'] = data['allestimento'].apply(categorize_trim)
    data['carrozzeria'] = data['carrozzeria'].apply(categorize_body)

    output_path = os.path.join(output_dir, output_filename)
    data.to_csv(output_path, index=False)

    return output_path

In [6]:
pathToRead = preprocess(df)
df = pd.read_csv(pathToRead)
df

,anno produzione,marca,modello,allestimento,carrozzeria,trasmissione,condizione,chilometraggio,colorazione,colore interni,prezzo
0,2015,kia,sorento,luxury,suv,automatic,5.0,16639.0,white,black,21500
1,2015,kia,sorento,luxury,suv,automatic,5.0,9393.0,white,beige,21500
2,2014,bmw,3 series,special edition,sedan,automatic,4.0,1331.0,gray,black,30000
3,2015,volvo,s60,other,sedan,automatic,4.0,14282.0,white,black,27750
4,2014,bmw,6 series gran coupe,other,sedan,automatic,4.0,2641.0,gray,black,67000
...,...,...,...,...,...,...,...,...,...,...,...
537709,2015,kia,k900,luxury,sedan,NaN,4.0,18255.0,silver,black,33000
537710,2012,ram,2500,other,pickup,automatic,5.0,54393.0,white,black,30800
537711,2012,bmw,x5,other,suv,automatic,5.0,50561.0,black,black,34000
537712,2015,nissan,altima,sport,sedan,automatic,4.0,16658.0,white,black,11100


In [7]:
from sklearn.model_selection import train_test_split
# Separazione in features e target
X = df.drop(columns=['prezzo'])
y = df['prezzo']

# Suddivisione in training e test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print('Valori mancanti train set: ' + str(X_train.isnull().sum().sum()))
print('Valori mancanti test set: ' + str(X_test.isnull().sum().sum()))

Valori mancanti train set: 80459
Valori mancanti test set: 34073


In [8]:
def data_preparation_train(X_train, y_train):
    calculate_modes_and_means(X_train)
    print("Shape before data_imputation:", X_train.shape[0])
    print("Shape of y_train:", y_train.shape[0])
    X_train = data_imputation(X_train)
    print("Shape after data_imputation:", X_train.shape[0])
    X_train = feature_engineer(X_train)
    print("Shape after feature_engineer:", X_train.shape[0])
    y_train = y_train[X_train.index]
    X_train = encode_train(X_train, y_train)
    X_train = normalize_train(X_train)
    X_train = remove_outliers(X_train)
    X_train = select_features_train(X_train)

    return X_train

In [9]:
def data_preparation_test(X_test):
    X_test = data_imputation(X_test)
    print("Shape before data_imputation:", X_test.shape[0])
    X_test = data_imputation(X_test)
    print("Shape after data_imputation:", X_test.shape[0])
    X_test = feature_engineer(X_test)
    print("Shape after feature_engineer:", X_test.shape[0])
    X_test = encode_test(X_test)
    X_test = normalize_test(X_test)
    X_test = select_features_test(X_test)

    return X_test

In [10]:
group_modes = {}
mean_condizione_by_anno = {}
mean_chilometraggio_by_anno = {}
mean_chilometraggio_by_eta = {}
mean_condizione_by_eta = {}
overall_modes = {}
overall_condizione_mean = None
overall_chilometraggio_mean = None

In [11]:
def calculate_modes_and_means(X_train):
    global group_modes, mean_condizione_by_anno, mean_chilometraggio_by_anno, mean_chilometraggio_by_eta, mean_condizione_by_eta, overall_modes, overall_condizione_mean, overall_chilometraggio_mean

    X = X_train.copy()
    group_modes['carrozzeria'] = X.groupby('modello')['carrozzeria'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['trasmissione'] = X.groupby('modello')['trasmissione'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['colorazione'] = X.groupby('marca')['colorazione'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['colore interni'] = X.groupby('marca')['colore interni'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['modello'] = X.groupby('marca')['modello'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['marca'] = X.groupby('modello')['marca'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()

    # Calcola le medie per anno
    mean_condizione_by_anno = X.groupby('anno produzione')['condizione'].mean().to_dict()
    mean_chilometraggio_by_anno = X.groupby('anno produzione')['chilometraggio'].mean().to_dict()

    mean_condizione_by_eta = {
        2015 - anno: media for anno, media in mean_condizione_by_anno.items()
    }

    mean_chilometraggio_by_eta = {
        2015 - anno: media for anno, media in mean_chilometraggio_by_anno.items()
    }

    # Calcola le mode complessive e le medie dopo imputazione
    overall_modes = {
        'carrozzeria': X['carrozzeria'].mode()[0],
        'trasmissione': X['trasmissione'].mode()[0],
        'colorazione': X['colorazione'].mode()[0],
        'colore interni': X['colore interni'].mode()[0],
        'modello': X['modello'].mode()[0],
        'marca': X['marca'].mode()[0]
    }

    # Calcola le medie complessive dopo imputazione per anno
    temp_condizione = X['condizione'].fillna(X['anno produzione'].map(mean_condizione_by_anno))
    overall_condizione_mean = temp_condizione.mean()

    temp_chilometraggio = X['chilometraggio'].fillna(X['anno produzione'].map(mean_chilometraggio_by_anno))
    overall_chilometraggio_mean = temp_chilometraggio.mean()


In [12]:
def data_imputation(X_train):
    global group_modes, mean_condizione_by_anno, mean_chilometraggio_by_anno, mean_chilometraggio_by_eta, mean_condizione_by_eta, overall_modes, overall_condizione_mean, overall_chilometraggio_mean

    X = X_train.copy()

    # Drop delle righe dove sia "Marca" che "Modello" sono null
    X = X.dropna(subset=["marca", "modello"], how="all")


    if 'trasmissione' in X.columns:
        X['trasmissione'] = X['trasmissione'].fillna(X['modello'].map(group_modes['trasmissione'])).fillna(
            overall_modes['trasmissione'])

    # Applica l'imputazione basata sui gruppi
    X['marca'] = X['marca'].fillna(X['marca'].map(group_modes['marca'])).fillna(overall_modes['marca'])
    X['modello'] = X['modello'].fillna(X['modello'].map(group_modes['modello'])).fillna(overall_modes['modello'])

    X['carrozzeria'] = X['carrozzeria'].fillna(X['modello'].map(group_modes['carrozzeria'])).fillna(overall_modes['carrozzeria'])

    X['colorazione'] = X['colorazione'].fillna(X['marca'].map(group_modes['colorazione'])).fillna(overall_modes['colorazione'])
    X['colore interni'] = X['colore interni'].fillna(X['marca'].map(group_modes['colore interni'])).fillna(overall_modes['colore interni'])

    X['allestimento'] = X['allestimento'].fillna('base')

    if('anno produzione' not in X.columns):
        # Imputa condizione e chilometraggio
        X['condizione'] = X['condizione'].fillna(X['età'].map(mean_condizione_by_eta)).fillna(overall_condizione_mean)
        X['chilometraggio'] = X['chilometraggio'].fillna(X['età'].map(mean_chilometraggio_by_eta)).fillna(overall_chilometraggio_mean)
    else:
        X['condizione'] = X['condizione'].fillna(X['anno produzione'].map(mean_condizione_by_anno)).fillna(
            overall_condizione_mean)
        X['chilometraggio'] = X['chilometraggio'].fillna(X['anno produzione'].map(mean_chilometraggio_by_anno)).fillna(
            overall_chilometraggio_mean)

    print("Duplicati prima eventuale drop")
    print(X.duplicated().sum().sum())
    X.drop_duplicates( inplace=True)
    print("Duplicati dopo eventuale drop")
    print(X.duplicated().sum().sum())

    return X

In [13]:
def feature_engineer(data):
    anno_rif = 2015
    data['età'] = anno_rif - data['anno produzione']
    data.drop(columns=['anno produzione'], inplace=True)

    return data

In [14]:
label_encoder = LabelEncoder()
catboost_encoder = ce.CatBoostEncoder()
marca_means = None
global_mean = None
known_makes = None
known_models = None
cat_cols = None

In [15]:
def encode_train(X_train, y_train):
    global marca_means, global_mean, known_makes, known_models, cat_cols, label_encoder, catboost_encoder
    
    if 'trasmissione' in X_train.columns:
        #Codifica 'trasmissione' con LabelEncoder
        X_train['trasmissione'] = label_encoder.fit_transform(X_train['trasmissione'])

    # Colonne categoriche da codificare con CatBoostEncoder
    cat_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns if col != 'trasmissione']
    
    # Fit CatBoostEncoder
    encoded = catboost_encoder.fit_transform(X_train[cat_cols], y_train)
    
    # Prepara sostituzioni per modelli non visti
    marca_means = X_train.join(y_train.rename('target')).groupby('marca')['target'].mean()
    global_mean = y_train.mean()
    known_makes = X_train['marca'].unique()
    known_models = X_train['modello'].unique()

    X_train[cat_cols] = encoded
    return X_train

In [16]:
def encode_test(X_test):
    global marca_means, global_mean, known_makes, known_models, cat_cols, label_encoder, catboost_encoder

    X = X_test.copy()

    if 'trasmissione' in X.columns:
        # Applica LabelEncoder a 'trasmissione'
        X['trasmissione'] = label_encoder.transform(X['trasmissione'])

    # Applica CatBoostEncoder alle altre colonne categoriche
    encoded = catboost_encoder.transform(X[cat_cols])

    # Gestisci modelli non presenti nel training set
    new_models_mask = ~X['modello'].isin(known_models)
    if new_models_mask.any():
        replacements = X.loc[new_models_mask, 'marca'].map(marca_means).fillna(global_mean)
        encoded.loc[new_models_mask, 'modello'] = replacements.values

    X[cat_cols] = encoded
    return X

In [17]:
scaler = StandardScaler()
numeric_cols = None

In [18]:
def normalize_train(X_train):
    global scaler, numeric_cols

    numeric_cols = X_train.select_dtypes(include=['number']).columns
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

    return X_train

In [19]:
def normalize_test(X_test):
    global scaler, numeric_cols

    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    return X_test

In [20]:
def remove_outliers(X_train):
    threshold = 3
    numeric_cols = X_train.select_dtypes(include=['number']).columns
    z_scores = (X_train[numeric_cols] - X_train[numeric_cols].mean()) / X_train[numeric_cols].std()
    mask = (z_scores.abs() <= threshold).all(axis=1)
    return X_train[mask]

In [21]:
threshold_variance = 0.1
features_to_keep = None

In [22]:
def select_features_train(X_train):
    global threshold_variance, features_to_keep
    variances = X_train.var()
    features_to_keep = variances[variances > threshold_variance].index.tolist()

    return X_train[features_to_keep]

In [23]:
def select_features_test(X_test):
    global features_to_keep

    return X_test[features_to_keep]

In [24]:
X_train = data_preparation_train(X_train, y_train)
y_train = y_train.loc[X_train.index]
X_test = data_preparation_test(X_test)
y_test = y_test.loc[X_test.index]

Shape before data_imputation: 376399
Shape of y_train: 376399
Duplicati prima eventuale drop
119
Duplicati dopo eventuale drop
0
Shape after data_imputation: 376280
Shape after feature_engineer: 376280
Duplicati prima eventuale drop
25
Duplicati dopo eventuale drop
0
Shape before data_imputation: 161290
Duplicati prima eventuale drop
0
Duplicati dopo eventuale drop
0
Shape after data_imputation: 161290
Shape after feature_engineer: 161290


In [25]:
X_train

,marca,modello,allestimento,carrozzeria,condizione,chilometraggio,colorazione,colore interni,età
370378,0.004477,0.003085,0.017218,0.006388,0.616314,-0.816936,0.009151,0.007220,-1.000578
229362,0.004477,0.003085,0.017218,0.006388,1.664729,-1.039917,0.009151,1.176733,-0.732552
387435,0.004477,0.003085,0.017218,1.058783,-0.432101,-0.613041,0.009151,2.480785,-1.000578
145008,0.004477,0.003085,0.464279,-0.440824,-1.480515,-0.937793,-2.660735,0.625968,-0.464525
106422,0.635450,0.003085,2.433350,0.184749,0.616314,-1.111151,2.867790,0.007220,-0.464525
...,...,...,...,...,...,...,...,...,...
137337,-0.614149,-0.427869,-1.195696,-0.717349,0.616314,0.066980,-0.426077,-0.118768,-0.732552
110268,-0.420629,-0.258052,-0.027667,0.978547,-1.480515,1.125937,-0.952311,-1.168103,0.607582
259178,1.620993,-1.130120,2.197728,-0.717336,0.616314,-0.209167,-0.952402,0.799910,1.411662
131932,1.620741,0.962786,2.197599,-0.717295,0.616314,-0.851036,1.155548,0.799894,-0.732552


In [26]:
X_test

,marca,modello,allestimento,carrozzeria,condizione,chilometraggio,colorazione,colore interni,età
143263,0.204156,0.018145,-0.027738,0.978459,1.664729,-0.866109,0.717702,-1.168139,-1.000578
336773,0.204156,0.399404,-0.027738,-0.717274,0.616314,-0.338872,-0.952423,0.799909,-1.000578
124520,0.204156,0.018145,-0.027738,0.978459,0.616314,-0.960990,0.717702,-1.168139,-1.000578
237169,-1.073470,-0.940634,-0.027738,-0.717274,0.616314,-0.478736,-0.426150,0.799909,-0.732552
402422,-0.420690,-0.753057,-0.027738,-1.500527,-0.432101,-0.077798,-0.426150,0.799909,-0.732552
...,...,...,...,...,...,...,...,...,...
390890,1.888237,0.600559,0.004678,-0.717274,0.616314,-0.747288,-0.952423,0.799909,-0.464525
448066,0.204156,0.800997,-0.027738,1.647064,-0.432101,0.960625,-0.426150,-1.168139,0.071528
177951,1.592313,-0.869141,-0.027738,-0.717274,1.664729,0.548489,0.091121,-1.168139,1.143635
102019,0.204156,-0.330050,-0.027738,-0.717274,-1.480515,1.302054,0.945719,3.052754,1.947715


In [27]:
X_train.isnull().sum()

marca             0
modello           0
allestimento      0
carrozzeria       0
condizione        0
chilometraggio    0
colorazione       0
colore interni    0
età               0
dtype: int64

In [28]:
X_test.isnull().sum()

marca             0
modello           0
allestimento      0
carrozzeria       0
condizione        0
chilometraggio    0
colorazione       0
colore interni    0
età               0
dtype: int64

In [ ]:
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

def random_forest_evaluation_depth(data, target_column='prezzo'):
    """
    Valuta l'effetto della profondità degli alberi (max_depth) su RandomForestRegressor.
    Traccia la loss di training e validazione in funzione della profondità.

    Parametri:
      - data: DataFrame contenente i dati.
      - target_column: Nome della colonna target (default 'Risultato').
    """

    train_losses = []
    val_losses = []
    depths = range(1, 40)

    for depth in depths:
        # Inizializzazione e training del modello per la profondità corrente
        model = RandomForestRegressor(
        max_depth=depth,                  # Controlla la complessità del modello
        n_estimators=300,               # Bilanciamento tra performance e velocità
        min_samples_split=10,            # Riduce overfitting
        min_samples_leaf=4,             # Previene overfitting sulle foglie
        max_features='sqrt',            # Ottimizza il trade-off bias-variance
        n_jobs=-1,                      # Usa tutti i core disponibili
        random_state=20,
        bootstrap=True,
        criterion= 'squared_error'
        )
        model.fit(X_train, y_train)

        # Calcolo della Mean Squared Error (MSE)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_loss = mean_squared_error(y_train, y_train_pred)
        val_loss = mean_squared_error(y_test, y_test_pred)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

    # Tracciamento dei risultati
    plt.figure(figsize=(10, 6))
    plt.plot(depths, train_losses, label='Training MSE', marker='o', linestyle='--', color='blue')
    plt.plot(depths, val_losses, label='Validation MSE', marker='o', linestyle='-', color='red')
    plt.xlabel('Tree Depth')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.title('Training & Validation Loss vs. Tree Depth')
    plt.legend()
    plt.grid(True)
    plt.show()

# Esegui la funzione
random_forest_evaluation_depth(df)